# 🧪 eTBF AI Diagnostics System - Patent Demonstration

## Enhanced Total Biological Fingerprint

**Inventor: Angel B. Yanes**

This notebook demonstrates the full multimodal diagnostic pipeline with real-world examples.

## Setup

### Prerequisites
- eTBF server running at `http://localhost:8000`
- Sample images in the `examples/` folder

In [ ]:
import requests
import json
import os
from pathlib import Path
from IPython.display import display, JSON, Image, HTML
import base64

API_URL = "http://localhost:8000/api"

In [ ]:
# Check if server is running
try:
    resp = requests.get(f"{API_URL}/health", timeout=5)
    print("✅ Server is running!")
    display(JSON(resp.json()))
except:
    print("❌ Server not running. Please start with: uvicorn app.main:app --host 0.0.0.0 --port 8000 --reload")

## Helper Function: Run a Diagnosis

In [ ]:
def run_diagnosis(patient_id: str, files_dict: dict) -> dict:
    """
    Run a diagnosis with the given patient ID and modality files.
    
    Args:
        patient_id: Patient identifier
        files_dict: Dict of {modality: file_path}
    
    Returns:
        Diagnosis result as dict
    """
    files = {}
    for modality, file_path in files_dict.items():
        if os.path.exists(file_path):
            files[modality] = (os.path.basename(file_path), open(file_path, 'rb'))
        else:
            print(f"⚠️ File not found: {file_path}")
    
    if not files:
        return {"error": "No valid files provided"}
    
    try:
        response = requests.post(
            f"{API_URL}/diagnose",
            data={"patient_id": patient_id},
            files=files,
            timeout=60
        )
        # Close file handles
        for _, (_, f) in files.items():
            f.close()
        return response.json()
    except Exception as e:
        return {"error": str(e)}

def display_diagnosis(result):
    """Pretty print diagnosis results."""
    if result.get("status") == "error":
        print(f"❌ Error: {result.get('detail', result.get('error', 'Unknown error'))}")
        return
    
    diagnosis = result.get("diagnosis", {})
    print("="*60)
    print("📊 DIAGNOSIS RESULT")
    print("="*60)
    print(f"🩺 Primary Diagnosis: {diagnosis.get('primary', 'N/A')}")
    print(f"📈 Confidence: {result.get('confidence', 0.0)*100:.1f}%")
    print(f"🔄 Differential: {', '.join(diagnosis.get('differential', []))}")
    
    temporal = diagnosis.get('temporal_context', {})
    if temporal:
        print(f"⏱️ Temporal Classification: {temporal.get('classification', 'N/A')} ({temporal.get('confidence', 0.0)*100:.0f}%)")
    
    print("\n🧠 Chain of Thought Reasoning:")
    print("-"*40)
    print(diagnosis.get('reasoning_trace', 'N/A')[:500] + "...")

    print("\n📋 Full Result:")
    display(JSON(result))

## Case 1: Aneurysmal Bone Cyst

**Source:** Radiopaedia - Aneurysmal bone cyst (ABC)

**Modalities:** X-ray, CT, Laboratory

In [ ]:
# Display sample images
def display_sample_images():
    html = '<div style="display:flex; gap:20px; flex-wrap:wrap;">'
    for name, path in [("X-ray", "examples/case_1_aneurysmal_bone_cyst/inputs/xray.jpg"),
                       ("CT", "examples/case_1_aneurysmal_bone_cyst/inputs/ct.jpg")]:
        if os.path.exists(path):
            html += f'<div><strong>{name}</strong><br><img src="{path}" style="max-width:300px; max-height:300px; border:1px solid #ddd; border-radius:8px;"/></div>'
        else:
            html += f'<div><strong>{name}</strong><br><span style="color:#999;">Image not found: {path}</span></div>'
    html += '</div>'
    display(HTML(html))

display_sample_images()

In [ ]:
# Run the diagnosis
result = run_diagnosis("Case_1_ABC", {
    "radiology": "examples/case_1_aneurysmal_bone_cyst/inputs/xray.jpg",
    "microscopy": "examples/case_1_aneurysmal_bone_cyst/inputs/ct.jpg",
    "laboratory": "examples/case_1_aneurysmal_bone_cyst/inputs/lab_results.json"
})

display_diagnosis(result)

In [ ]:
# Save result for patent proof
output_dir = Path("examples/case_1_aneurysmal_bone_cyst/outputs")
output_dir.mkdir(parents=True, exist_ok=True)
with open(output_dir / "diagnosis_result.json", "w") as f:
    json.dump(result, f, indent=2)
print("✅ Result saved to examples/case_1_aneurysmal_bone_cyst/outputs/diagnosis_result.json")

## Case 2: Pulmonary Nodule

**Source:** Open-i / Radiopaedia

**Modalities:** CT, Laboratory

In [ ]:
result = run_diagnosis("Case_2_Nodule", {
    "radiology": "examples/case_2_pulmonary_nodule/inputs/ct.jpg",
    "laboratory": "examples/case_2_pulmonary_nodule/inputs/lab_results.json"
})

display_diagnosis(result)

## Case 3: Multiple Myeloma

**Source:** Radiopaedia

**Modalities:** Electrophoresis, X-ray, Laboratory

In [ ]:
result = run_diagnosis("Case_3_Myeloma", {
    "electrophoresis": "examples/case_3_multiple_myeloma/inputs/gel.jpg",
    "radiology": "examples/case_3_multiple_myeloma/inputs/xray.jpg",
    "laboratory": "examples/case_3_multiple_myeloma/inputs/lab_results.json"
})

display_diagnosis(result)

## Case 4: Acute Gouty Arthritis

**Source:** Radiopaedia - Acute gouty arthritis

**Modalities:** X-ray, Ultrasound, Microbiology, Laboratory

In [ ]:
result = run_diagnosis("Case_4_Gout", {
    "radiology": "examples/case_4_acute_gout/inputs/xray.jpg",
    "ultrasound": "examples/case_4_acute_gout/inputs/ultrasound.jpg",
    "laboratory": "examples/case_4_acute_gout/inputs/lab_results.json"
})

display_diagnosis(result)

## Case 5: Papillary Thyroid Carcinoma

**Source:** Radiopaedia - Papillary thyroid carcinoma

**Modalities:** Ultrasound, Histopathology, Laboratory

In [ ]:
result = run_diagnosis("Case_5_Thyroid", {
    "ultrasound": "examples/case_5_papillary_thyroid_carcinoma/inputs/ultrasound.jpg",
    "laboratory": "examples/case_5_papillary_thyroid_carcinoma/inputs/lab_results.json"
})

display_diagnosis(result)

## Case 6: Solitary Fibrous Tumor

**Source:** Radiopaedia - Solitary fibrous tumour

**Modalities:** MRI, Histopathology, Laboratory

In [ ]:
result = run_diagnosis("Case_6_SFT", {
    "radiology": "examples/case_6_solitary_fibrous_tumor/inputs/mri.jpg",
    "laboratory": "examples/case_6_solitary_fibrous_tumor/inputs/lab_results.json"
})

display_diagnosis(result)

## Summary of Results

This notebook demonstrates the eTBF AI Diagnostics System's ability to:

1. ✅ **Ingest multiple data modalities** (X-ray, CT, MRI, Ultrasound, Electrophoresis, Histopathology, Microbiology, Laboratory)
2. ✅ **Extract features** from each modality using specialized encoders
3. ✅ **Retrieve similar cases** from the vector database
4. ✅ **Fuse multimodal information** for comprehensive diagnosis
5. ✅ **Generate explainable outputs** with reasoning traces
6. ✅ **Classify temporal context** (acute/subacute/chronic)

## Cases Demonstrated

| Case | Modalities | Expected Diagnosis |
|------|------------|-------------------|
| Case 1 | X-ray, CT, Laboratory | Aneurysmal Bone Cyst |
| Case 2 | CT, Laboratory | Pulmonary Nodule |
| Case 3 | Electrophoresis, X-ray, Laboratory | Multiple Myeloma |
| Case 4 | X-ray, Ultrasound, Microbiology, Laboratory | Acute Gout |
| Case 5 | Ultrasound, Histopathology, Laboratory | Papillary Thyroid Carcinoma |
| Case 6 | MRI, Histopathology, Laboratory | Solitary Fibrous Tumor |

---

**Patent Pending** - Inventor: Angel B. Yanes